In [1]:
import os
import shutil
import numpy as np

# Point these at your actual folders
DATA_ROOT = r"D:\Sign Lang Landmarks\processed"          # existing train/val/test
OUT_ROOT  = r"D:\Sign Lang Landmarks\processed_resplit"  # new output goes here

SPLITS_IN = ["train", "val", "test"]
RESAMPLE_LEN = 50   # frames we resample every sequence to, just for comparison
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

rng = np.random.default_rng(42)   # seeded, so re-runs give the same split

In [2]:
def resample_seq(arr, n=RESAMPLE_LEN):
    """Stretch or squeeze a (T, 291) sequence to exactly (n, 291) frames."""
    T = arr.shape[0]
    if T == n:
        return arr
    idx = np.linspace(0, T - 1, n)          # n evenly spaced positions along the original timeline
    idx0 = np.floor(idx).astype(int)        # frame just before each position
    idx1 = np.minimum(idx0 + 1, T - 1)      # frame just after (clamped so it doesn't run off the end)
    frac = (idx - idx0)[:, None]            # how far between idx0 and idx1 we are (0 to 1)
    return arr[idx0] * (1 - frac) + arr[idx1] * frac   # linear blend between the two frames


def load_and_vectorize(path):
    """Load a clip, resample it, flatten it into one vector, and normalize it."""
    arr = np.load(path).astype(np.float32)
    arr = resample_seq(arr)
    v = arr.flatten()                       # (50, 291) -> (14550,) single vector
    norm = np.linalg.norm(v) + 1e-8         # +1e-8 avoids divide-by-zero on a flat/empty clip
    return v / norm

In [3]:
# Pick one class to test with
test_class = "Yes"

# Gather every file for this class across train/val/test
all_files = []
for sp in SPLITS_IN:
    cls_dir = os.path.join(DATA_ROOT, sp, test_class)
    if os.path.isdir(cls_dir):
        for f in os.listdir(cls_dir):
            if f.endswith(".npy"):
                all_files.append(os.path.join(cls_dir, f))

print(f"Total files for '{test_class}': {len(all_files)}")

# Split into originals (real recordings) vs augmented (generated variants)
originals  = [f for f in all_files if not os.path.basename(f).startswith("aug_")]
augmenteds = [f for f in all_files if os.path.basename(f).startswith("aug_")]

print(f"Originals: {len(originals)}, Augmented: {len(augmenteds)}")

Total files for 'Yes': 291
Originals: 126, Augmented: 165


In [4]:
# Vectorize all originals once
orig_vecs = np.stack([load_and_vectorize(f) for f in originals])   # shape: (n_originals, 14550)
orig_keys = originals   # keeps track of which row = which file

# group_of maps every file -> the "source" it belongs to
group_of = {f: f for f in originals}   # each original is its own source

for af in augmenteds:
    v = load_and_vectorize(af)
    sims = orig_vecs @ v              # cosine similarity to every original at once (dot product, since both are normalized)
    best_idx = np.argmax(sims)        # index of the most similar original
    group_of[af] = orig_keys[best_idx]

In [5]:
for af in augmenteds[:5]:
    print(os.path.basename(af), "->", os.path.basename(group_of[af]))

aug_Yes_0.npy -> A_Yes_10.npy
aug_Yes_1.npy -> A_Yes_100.npy
aug_Yes_10.npy -> A_Yes_117.npy
aug_Yes_100.npy -> A_Yes_119.npy
aug_Yes_101.npy -> A_Yes_12.npy


In [6]:
# Build groups: each original + whichever aug_ files matched to it
groups = {}
for f, g in group_of.items():
    groups.setdefault(g, []).append(f)

print(f"Number of groups (= number of originals): {len(groups)}")

# Peek at one group to see what it looks like
sample_key = list(groups.keys())[0]
print(f"\nExample group '{os.path.basename(sample_key)}':")
for f in groups[sample_key]:
    print(" ", os.path.basename(f))

Number of groups (= number of originals): 126

Example group 'A_Yes_10.npy':
  A_Yes_10.npy
  aug_Yes_0.npy
  aug_Yes_88.npy


In [7]:
group_items = list(groups.items())
rng.shuffle(group_items)          # shuffle at the GROUP level, not file level

n = len(group_items)
n_train = int(round(n * TRAIN_FRAC))
n_val   = int(round(n * VAL_FRAC))

split_assignment = {}   # file path -> "train" / "val" / "test"
for i, (source, files) in enumerate(group_items):
    if i < n_train:
        sp = "train"
    elif i < n_train + n_val:
        sp = "val"
    else:
        sp = "test"
    for f in files:
        split_assignment[f] = sp

# quick count check
from collections import Counter
print(Counter(split_assignment.values()))

Counter({'train': 203, 'val': 44, 'test': 44})


In [8]:
all_classes = sorted(os.listdir(os.path.join(DATA_ROOT, "train")))
print(f"Total classes: {len(all_classes)}")

global_split_assignment = {}   # file path -> "train"/"val"/"test", across ALL classes

for cls in all_classes:
    # 1. gather files for this class
    cls_files = []
    for sp in SPLITS_IN:
        cls_dir = os.path.join(DATA_ROOT, sp, cls)
        if os.path.isdir(cls_dir):
            for f in os.listdir(cls_dir):
                if f.endswith(".npy"):
                    cls_files.append(os.path.join(cls_dir, f))

    originals  = [f for f in cls_files if not os.path.basename(f).startswith("aug_")]
    augmenteds = [f for f in cls_files if os.path.basename(f).startswith("aug_")]

    # 2. match aug files to nearest original
    group_of = {f: f for f in originals}
    if originals:
        orig_vecs = np.stack([load_and_vectorize(f) for f in originals])
        orig_keys = originals
        for af in augmenteds:
            v = load_and_vectorize(af)
            sims = orig_vecs @ v
            group_of[af] = orig_keys[int(np.argmax(sims))]
    else:
        for af in augmenteds:
            group_of[af] = af   # no originals — fall back to treating it standalone

    # 3. build groups
    groups = {}
    for f, g in group_of.items():
        groups.setdefault(g, []).append(f)

    # 4. shuffle + split by group
    group_items = list(groups.items())
    rng.shuffle(group_items)
    n = len(group_items)
    n_train = int(round(n * TRAIN_FRAC))
    n_val   = int(round(n * VAL_FRAC))

    for i, (source, files) in enumerate(group_items):
        sp = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
        for f in files:
            global_split_assignment[f] = sp

    print(f"{cls:20s}: {len(originals):3d} orig, {len(augmenteds):3d} aug, {n:3d} groups")

Total classes: 52
Ambulance           : 144 orig, 147 aug, 144 groups
Bad                 : 132 orig, 159 aug, 132 groups
Bandage             : 140 orig, 151 aug, 140 groups
Book                : 225 orig,  66 aug, 225 groups
Come                : 133 orig, 158 aug, 133 groups
Cough               : 158 orig, 133 aug, 158 groups
Doctor              : 165 orig, 126 aug, 165 groups
Eat                 : 234 orig,  57 aug, 234 groups
Elder Brother       : 158 orig, 133 aug, 158 groups
Elder Sister        : 151 orig, 140 aug, 151 groups
Father              : 151 orig, 140 aug, 151 groups
Fever               : 174 orig, 117 aug, 174 groups
Friday              : 227 orig,  64 aug, 227 groups
Go                  : 224 orig,  67 aug, 224 groups
Good                : 141 orig, 150 aug, 141 groups
Headache            : 140 orig, 151 aug, 140 groups
Help                : 169 orig, 122 aug, 169 groups
Here                : 139 orig, 152 aug, 139 groups
Home                : 231 orig,  60 aug, 231 g

In [9]:
copied = 0
for f, sp in global_split_assignment.items():
    cls = os.path.basename(os.path.dirname(f))          # class name = parent folder name
    dest_dir = os.path.join(OUT_ROOT, sp, cls)
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy2(f, os.path.join(dest_dir, os.path.basename(f)))
    copied += 1

print(f"Copied {copied} files into {OUT_ROOT}")

Copied 15132 files into D:\Sign Lang Landmarks\processed_resplit


In [10]:
for sp in ["train", "val", "test"]:
    sp_path = os.path.join(OUT_ROOT, sp)
    n_classes = len(os.listdir(sp_path))
    n_files = sum(len(os.listdir(os.path.join(sp_path, c))) for c in os.listdir(sp_path))
    print(f"{sp}: {n_classes} classes, {n_files} files")

train: 52 classes, 9601 files
val: 52 classes, 2227 files
test: 52 classes, 2222 files
